# Movement-strategy validation

**Question.** How much does quantising the commanded direction degrade
reproduction of a commanded path, and how much of the previously reported
difference was actually the kinematic model?

**Scope.** Standalone; the control strategy only. Everything is in *controller
units* (command counts). Actuator calibration - servo ticks, spool radius, dead
zone, latency - is out of scope and belongs to `validetion/Servomotor` and
`validetion/Servo+thimble`. Do not mix those numbers in here.

**Design - a full 3 x 2 factorial.** Strategy and kinematic model are separate
axes, and this study crosses them:

| | Planar model | IK model |
|---|---|---|
| `CARDINAL` (4-way) | . | . |
| `CARDINAL_DIAGONAL` (8-way) | . | . |
| `FREE_FORM` (continuous) | . | . |

Reading *down a column* gives the quantisation effect at a fixed model.
Reading *across a row* gives the model effect at a fixed strategy.

`MovementStrategy.IK` does not appear as a row because it is not a strategy:
it is `FREE_FORM` paired with the IK model, i.e. the bottom-right cell. The
historic "CD vs IK" comparison was the off-diagonal (CD/planar against
free-form/IK), which is why its result could not be attributed to either cause.

**Limitation.** Each model is decoded by its own inverse - trilateration for
planar, wire FK for IK - so this measures strategy- and model-induced path
error, not physical mechanism accuracy. That FK genuinely inverts the
controller's IK path is verified in `tests/test_wire_forward_kinematics.py`.

All computation lives in `analysis.py` and `figures.py`; this notebook only
calls them, so there is exactly one implementation.

In [1]:
import pandas as pd

from analysis import StudyConfig, effect_summary, run_study
from figures import plot_error_decomposition, plot_motor_commands, plot_reconstructed_circles

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

config = StudyConfig()
samples, metrics = run_study(config)

cells = len(config.strategies) * len(config.kinematic_models)
print(f"{len(samples) // cells} samples per cell, radius {config.radius:.0f} controller units, "
      f"{len(config.strategies)} strategies x {len(config.kinematic_models)} models.")

301 samples per cell, radius 160 controller units, 3 strategies x 2 models.


## The two effects, separated

    ideal target --(strategy quantisation)--> quantised target
                 --(model + integer truncation)--> reconstructed point

`quantisation_rms` is the strategy's intrinsic cost and is identical in both
models by construction - a useful check that nothing else is varying.
`execution_rms` is what the model and integer truncation cost.

In [2]:
metrics[["quantisation_rms", "quantisation_max",
         "execution_rms", "execution_max",
         "total_rms", "total_max"]].round(3)

quantisation_rms  quantisation_max  execution_rms  execution_max  total_rms  total_max
kinematic_model strategy                                                                                                 
planar          cardinal                     61.067           120.345          0.706          2.687     60.962    120.352
                cardinal_diagonal            32.644            72.479          0.769          2.687     32.589     72.603
                free_form                     0.000             0.000          0.824          1.534      0.824      1.534
ik              cardinal                     61.067           120.345          2.754          7.039     60.865    121.848
                cardinal_diagonal            32.644            72.479          2.956          7.039     32.612     74.045
                free_form                     0.000             0.000          3.517         10.426      3.517     10.426

In [3]:
print("Total RMS error: strategy (rows) x model (columns)")
effect_summary(metrics).round(2)

Total RMS error: strategy (rows) x model (columns)


model,ik,planar
strategy,,
cardinal,60.87,60.96
cardinal_diagonal,32.61,32.59
free_form,3.52,0.82


## Command effort is a model effect

This is the finding that was previously attributed to the strategy. Holding the
strategy fixed and switching the model changes command amplitude by ~3.8x;
holding the model fixed and changing the strategy barely moves it. The IK
mechanism transmits far less cable travel per unit of tactor motion than the
planar straight-line approximation assumes.

The one genuine strategy signature in the commands is `max_step_jump` - the
discontinuity when a quantised direction snaps to a new sector.

In [4]:
metrics[["peak_abs_command", "rms_command",
         "total_motor_travel", "max_step_jump"]].round(3)

peak_abs_command  rms_command  total_motor_travel  max_step_jump
kinematic_model strategy                                                                           
planar          cardinal                      160.0      100.168              2080.0          223.0
                cardinal_diagonal             160.0      100.273              2416.0          121.0
                free_form                     160.0      100.318              2420.0            5.0
ik              cardinal                       45.0       26.484               577.0           72.0
                cardinal_diagonal              50.0       26.397               603.0           45.0
                free_form                      53.0       26.365               622.0            2.0

## Shape metrics, and why they mislead

Quantisation is radius-preserving, so every reconstructed point sits on the
commanded radius. Aspect ratio and circle-fit RMS therefore score all three
strategies as near-perfect circles while the point-to-point error differs by
more than an order of magnitude. That is why this study reports point-to-point
error as its primary metric.

In [5]:
metrics[["radial_rms", "circle_fit_rms", "aspect_ratio",
         "closure_error", "decode_invalid_steps"]].round(3)

radial_rms  circle_fit_rms  aspect_ratio  closure_error  decode_invalid_steps
kinematic_model strategy                                                                                        
planar          cardinal                0.602           0.108         1.001          2.687                     0
                cardinal_diagonal       0.699           0.161         1.001          2.687                     0
                free_form               0.745           0.257         1.001          0.000                     0
ik              cardinal                1.912           0.323         1.004          2.853                     0
                cardinal_diagonal       2.167           0.768         1.004          2.853                     0
                free_form               3.143           1.693         0.997          0.000                     0

## Figures

In [6]:
for model in config.kinematic_models:
    print("saved", plot_reconstructed_circles(samples, config, None, model.value).name)
print("saved", plot_error_decomposition(samples, metrics, config).name)
print("saved", plot_motor_commands(samples, config).name)

saved reconstructed_circles_planar.png


saved reconstructed_circles_ik.png


saved error_decomposition.png


saved motor_commands_by_strategy.png
